# Theory-Driven GEPA Experiment: Exploration Notebook

This notebook explores results from the pre-registered low-data experiment.

**Research question:** Does a typed Popperian critic improve DSPy optimization in the low-data regime?

**Arms:** A0 (zero-shot), A (GEPA), E (check-only), C (untyped critic), B (Popperian), D (Millian), F (SIMBA)

In [ ]:
import sys
sys.path.insert(0, '/workspace')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from experiments.store import ResultsStore
from experiments.analysis import compare_arms, check_refutation_criteria, summarize_results

sns.set_style('whitegrid')
%matplotlib inline

## Load Results

In [ ]:
# Update this path to your results directory
RESULTS_DIR = Path('/workspace/results')

store = ResultsStore(RESULTS_DIR, compress=True)
results = list(store.load_runs())

print(f"Loaded {len(results)} runs")
print(f"Arms: {sorted(set(r.arm for r in results))}")
print(f"N values: {sorted(set(r.n_labels for r in results))}")
print(f"Seeds: {sorted(set(r.seed for r in results))}")

## Filter to Valid Runs

Exclude runs that exceeded budget tolerance.

In [ ]:
valid_results = [r for r in results if r.within_budget_tolerance and r.optimizer_success]
print(f"Valid runs: {len(valid_results)} / {len(results)}")

# Show failures
failures = [r for r in results if not r.optimizer_success]
if failures:
    print(f"\nFailed runs: {len(failures)}")
    for r in failures:
        print(f"  {r.arm} N={r.n_labels} seed={r.seed}: {r.optimizer_error}")

## Summary Statistics

In [ ]:
summary = summarize_results(valid_results)
summary_df = pd.DataFrame(summary).T
summary_df

## Accuracy by Arm and N

In [ ]:
# Prepare data for plotting
plot_data = []
for r in valid_results:
    plot_data.append({
        'arm': r.arm,
        'N': r.n_labels,
        'seed': r.seed,
        'test_accuracy': r.test_accuracy * 100
    })

df = pd.DataFrame(plot_data)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=df, x='N', y='test_accuracy', hue='arm', marker='o', ax=ax)
ax.set_xlabel('Number of Labels')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Test Accuracy by Arm and N')
ax.legend(title='Arm')
plt.tight_layout()
plt.show()

## Refutation Criteria (R1-R5)

Pre-registered hypothesis tests with 95% paired bootstrap CIs.

In [ ]:
refutation_results = check_refutation_criteria(valid_results)

for criterion, result in refutation_results.items():
    print(f"\n{criterion}")
    print("=" * 60)
    
    if 'error' in result:
        print(f"ERROR: {result['error']}")
        continue
    
    print(result['criterion'])
    
    if 'mean_diff' in result:
        print(f"  Mean difference: {result['mean_diff']:.2f} points")
        print(f"  95% CI: [{result['ci_lower']:.2f}, {result['ci_upper']:.2f}]")
    
    if 'diff_n10' in result:
        print(f"  Δ at N=10: {result['diff_n10']:.2f} points")
        print(f"  Δ at N=100: {result['diff_n100']:.2f} points")
    
    print(f"  Refuted: {result['refuted']}")
    print(f"  {result['interpretation']}")

## Pairwise Comparisons

Compare specific arms with bootstrap CIs.

In [ ]:
comparisons = [
    ('B', 'A', 10),
    ('B', 'A', 25),
    ('B', 'E', None),  # pooled small-N
    ('B', 'C', None),
    ('B', 'F', None),
]

for arm_x, arm_y, n in comparisons:
    try:
        result = compare_arms(valid_results, arm_x, arm_y, n_labels=n)
        
        n_label = f"N={n}" if n else "pooled"
        print(f"\n{arm_x} vs {arm_y} ({n_label})")
        print(f"  {arm_x} mean: {result.mean_x:.2f}%")
        print(f"  {arm_y} mean: {result.mean_y:.2f}%")
        print(f"  Difference: {result.mean_diff:.2f} points")
        print(f"  95% CI: [{result.ci_lower:.2f}, {result.ci_upper:.2f}]")
        print(f"  ({result.n_seeds} seeds × {result.n_test_items} items)")
    except ValueError as e:
        print(f"\n{arm_x} vs {arm_y} ({n_label}): {e}")

## Budget Analysis

In [ ]:
budget_data = []
for r in valid_results:
    budget_data.append({
        'arm': r.arm,
        'N': r.n_labels,
        'seed': r.seed,
        'program_rollouts': r.program_rollouts,
        'reflection_calls': r.reflection_calls,
        'reflection_tokens': r.reflection_tokens,
    })

budget_df = pd.DataFrame(budget_data)

# Mean by arm
budget_summary = budget_df.groupby('arm').agg({
    'program_rollouts': ['mean', 'std'],
    'reflection_calls': ['mean', 'std'],
    'reflection_tokens': ['mean', 'std']
}).round(1)

print("\nBudget usage by arm:")
budget_summary

## Proposer Logs (Arms B, C, D, E)

Inspect critic behavior for arms with custom proposers.

In [ ]:
# Example: look at Popperian proposer logs (arm B)
arm_b_runs = [r for r in valid_results if r.arm == 'B' and r.proposer_log]

if arm_b_runs:
    print(f"Found {len(arm_b_runs)} arm B runs with proposer logs\n")
    
    # Take one example
    example = arm_b_runs[0]
    print(f"Example: N={example.n_labels}, seed={example.seed}")
    print(f"Proposer log entries: {len(example.proposer_log)}")
    
    # Show first few entries
    for i, entry in enumerate(example.proposer_log[:3]):
        print(f"\nEntry {i+1}:")
        print(f"  Attempt: {entry.get('attempt')}")
        print(f"  Refutations: {entry.get('n_refutations')}")
        print(f"  Survived: {entry.get('survived')}")
        print(f"  Checked IDs: {entry.get('checked_ids')}")
else:
    print("No arm B runs with proposer logs found")

## Export Results

Save summary tables for the paper.

In [ ]:
# Save summary to CSV
summary_df.to_csv(RESULTS_DIR / 'summary.csv')
budget_summary.to_csv(RESULTS_DIR / 'budget_summary.csv')

print("Saved summary.csv and budget_summary.csv")